In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np
import random
import joblib
from pathlib import Path
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define the SGD Base Rates (Scaled down from VND)
SUBTYPE_BASE_SGD = {
    "DEEP_CLEANING": (45, 150),
    "APARTMENT_CLEANING": (28, 90),
    "HOUSE_CLEANING": (35, 120),
    "OFFICE_CLEANING": (50, 160),
    "KITCHEN_BATHROOM_CLEANING": (30, 95),
    "BASIC_CLEANING": (18, 65),
    "URGENT_DOCUMENT_DELIVERY": (18, 55),
    "DOCUMENT_DELIVERY": (9, 28),
    "PARCEL_DELIVERY": (12, 42),
    "PLUMBING_REPAIR": (25, 85),
    "ELECTRICAL_REPAIR": (30, 100),
    "GENERAL_REPAIR": (22, 70),
    "HEAVY_FURNITURE_MOVING": (90, 280),
    "SMALL_MOVING": (35, 120),
    "ERRANDS": (15, 55),
    "SUBJECT_TUTORING": (25, 100),
    "GENERAL_HELP": (18, 70),
}

def generate_sgd_dataset(size=1500):
    random.seed(2026)
    data = []
    
    for _ in range(size):
        subtype = random.choice(list(SUBTYPE_BASE_SGD.keys()))
        category = subtype.split('_')[-1] if '_' in subtype else "OTHER"
        
        # Generate the 8 required features for the API
        distance_km = random.uniform(1.0, 25.0)
        duration_hours = random.uniform(0.5, 6.0)
        urgency_score = random.choice([0.2, 0.25, 0.3, 0.7, 1.0])
        is_urgent_flag = 1 if urgency_score >= 0.8 else 0
        complexity_score = random.uniform(0.3, 0.95)
        quantity_score = random.uniform(0.3, 0.95)
        scope_score = random.uniform(0.3, 0.95)
        location_score = random.uniform(0.4, 0.9)
        
        # Calculate SGD Price based on multipliers
        base_min, base_max = SUBTYPE_BASE_SGD[subtype]
        midpoint = (base_min + base_max) / 2
        
        multiplier = (
            0.72
            + complexity_score * 0.25
            + quantity_score * 0.15
            + scope_score * 0.14
            + location_score * 0.08
            + min(distance_km, 30.0) / 150.0
            + (0.14 if is_urgent_flag else 0.0)
        )
        
        # Add random market variance (+/- 10%)
        variance = random.uniform(0.9, 1.1)
        target_price = round(max(12.0, midpoint * multiplier * variance))
        
        # Generate text feature exactly as the API formats it
        text_feature = f"title sample task description task for {subtype.lower()} subtype {subtype.lower()} category {category.lower()}"
        
        data.append([
            text_feature, distance_km, duration_hours, is_urgent_flag, urgency_score, 
            complexity_score, quantity_score, scope_score, location_score, target_price
        ])
        
    cols = ['text_feature', 'distance_km', 'duration_hours', 'is_urgent_flag', 'urgency_score', 
            'complexity_score', 'quantity_score', 'scope_score', 'location_score', 'target_price']
    return pd.DataFrame(data, columns=cols)

# 2. Create the dataset and save the CSV
df = generate_sgd_dataset(1500)
df.to_csv('../data/task_pricing_data_sgd.csv', index=False)
print("Saved task_pricing_data_sgd.csv to /data folder.")

# 3. Train the Models
print("Training models...")
vectorizer = TfidfVectorizer(max_features=320, ngram_range=(1, 2))
X_text = vectorizer.fit_transform(df['text_feature']).toarray()
X_num = df[['distance_km', 'duration_hours', 'is_urgent_flag', 'urgency_score', 
            'complexity_score', 'quantity_score', 'scope_score', 'location_score']].values

X_all = np.hstack([X_text, X_num])
y = df['target_price'].values

gbr = GradientBoostingRegressor(random_state=2026, n_estimators=260, max_depth=3)
rf = RandomForestRegressor(random_state=2026, n_estimators=220, max_depth=12, n_jobs=-1)

gbr.fit(X_all, y)
rf.fit(X_all, y)

# 4. Export the Model Bundle for the FastAPI Server
bundle = {
    "vectorizer": vectorizer,
    "gbr": gbr,
    "rf": rf,
    "mode": "pretrained_ensemble",
    "model_name": "GradientBoosting + RandomForest (SGD)"
}

model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)
joblib.dump(bundle, model_dir / 'pricing_bundle.pkl')
print("Successfully exported pricing_bundle.pkl to /models folder.")

Saved task_pricing_data_sgd.csv to /data folder.
Training models...
Successfully exported pricing_bundle.pkl to /models folder.
